# 🌱 Beginner Project 3: Data Manipulation & Aggregation

Welcome to **Beginner Project 3**! In this project, you will master advanced data manipulation, feature creation, group-by aggregations, and multi-metric analysis using `pandas`.

### 🎯 Objectives
- Load a balanced dataset of positive and negative reviews from Hugging Face (`stanfordnlp/imdb`).
- Engineer text-based features (character count, word count, uppercase ratio, punctuation count).
- Filter and subset data based on complex criteria.
- Perform group-by operations and compute statistical summaries (mean, median, std) across sentiments.
- Build pivot tables and plot summary metrics.


In [ ]:
# Step 1: Install required dependencies
!pip install -q datasets pandas matplotlib


In [ ]:
# Step 2: Import libraries
import pandas as pd
import matplotlib.pyplot as plt
from datasets import load_dataset

plt.style.use('ggplot')
print("Libraries imported successfully!")


In [ ]:
# Step 3: Load balanced dataset from Hugging Face
# Load negative (label 0) and positive (label 1) reviews to form a balanced dataset
neg_dataset = load_dataset("stanfordnlp/imdb", split="train[:250]")
pos_dataset = load_dataset("stanfordnlp/imdb", split="train[12500:12750]")

df_neg = pd.DataFrame(neg_dataset)
df_pos = pd.DataFrame(pos_dataset)
df = pd.concat([df_neg, df_pos], ignore_index=True)

# Map sentiment labels (0 = Negative, 1 = Positive)
df['sentiment'] = df['label'].map({0: 'Negative', 1: 'Positive'})

print("Dataset shape:", df.shape)
print("Sentiment distribution:\n", df['sentiment'].value_counts())
df.head()


In [ ]:
# Step 4: Feature Engineering
# Create feature: Character length
df['char_count'] = df['text'].apply(len)

# Create feature: Word count
df['word_count'] = df['text'].apply(lambda t: len(t.split()))

# Create feature: Average word length
df['avg_word_length'] = df['char_count'] / df['word_count']

# Create feature: Uppercase letter ratio (indicative of emphasis/shouting in reviews)
df['uppercase_ratio'] = df['text'].apply(lambda t: sum(1 for c in t if c.isupper()) / max(len(t), 1))

df[['text', 'sentiment', 'word_count', 'avg_word_length', 'uppercase_ratio']].head()


In [ ]:
# Step 5: Data Filtering and Subsetting
# Filter 1: Find long reviews with high uppercase ratio
long_shouting_reviews = df[(df['word_count'] > 200) & (df['uppercase_ratio'] > 0.04)]
print(f"Number of long reviews with high uppercase ratio: {len(long_shouting_reviews)}")

# Filter 2: Compare positive vs negative review counts in filtered subset
print("\nSentiment distribution in subset:")
print(long_shouting_reviews['sentiment'].value_counts())


In [ ]:
# Step 6: GroupBy Aggregations
# Group by sentiment and calculate mean, median, and std for text metrics
aggregated_stats = df.groupby('sentiment').agg({
    'word_count': ['mean', 'median', 'std'],
    'char_count': ['mean', 'median'],
    'uppercase_ratio': ['mean', 'std']
})

print("--- Aggregated Text Statistics by Sentiment ---")
aggregated_stats


In [ ]:
# Step 7: Pivot Table & Visualization
# Create pivot table for sentiment vs binned word length category
df['review_length_cat'] = pd.cut(df['word_count'], bins=[0, 100, 250, 1000], labels=['Short', 'Medium', 'Long'])

pivot_tb = pd.crosstab(df['review_length_cat'], df['sentiment'])
print("--- Pivot Table: Review Length Category vs Sentiment ---")
print(pivot_tb)

# Plotting the pivot table
pivot_tb.plot(kind='bar', color=['#e74c3c', '#2ecc71'], edgecolor='black', figsize=(8, 5))
plt.title("Review Length Distribution by Sentiment Class", fontsize=14, fontweight='bold')
plt.xlabel("Review Length Category", fontsize=12)
plt.ylabel("Count", fontsize=12)
plt.xticks(rotation=0)
plt.legend(title="Sentiment")
plt.tight_layout()
plt.show()
